In [1]:
# Core imports shared by all examples
from __future__ import annotations

from typing import Any, List

from IPython.display import Image, display
from langgraph.graph import StateGraph, END , START
from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from openinference.instrumentation.langchain import LangChainInstrumentor
from typing_extensions import Optional, Annotated, List, Sequence

from opentelemetry import trace

import rich
from langgraph.graph.message import add_messages

import operator


In [2]:
import dotenv
import os

In [3]:
dotenv.load_dotenv("../env_workshop")

True

In [5]:
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")
PHOENIX_PROJECT_NAME=os.environ.get("PHOENIX_PROJECT_NAME")


In [6]:
from phoenix.otel import register
from openinference.instrumentation import using_metadata

# configure the Phoenix tracer
tracer_provider = register(
  project_name=PHOENIX_PROJECT_NAME, 
  auto_instrument=False 
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

tracer = trace.get_tracer(__name__)


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: npatta01
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: llm-tracing.np-training.dev:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [7]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=OPENAI_BASE_URL,
    #temperature=0.2,
    #max_tokens=512,
)

In [ ]:
@tool
def animal_joke(animal: str) -> str:
    """Return a short, clean joke for turtle/cat/dog (falls back to dog)."""

    JOKES = {
    "turtle": "Why don’t turtles use public Wi-Fi? Too many shell networks.",
    "cat":    "Why did the cat sit on the computer? To keep an eye on the mouse.",
    "dog":    "What do you call a dog that does magic? A labracadabrador.",
    }
    return JOKES.get((animal or "").lower().strip(), JOKES["dog"])


@tool
def fetch_cute_animals() -> str:
    """Fetch a list of cute animals."""
    return ["dog", "cat", "turtle"]

tool_model = model.bind_tools([animal_joke])  # for tool calling


# ---- Build messages explicitly (System + Human) ----
animal = "turtle"
messages = [
    SystemMessage(content="You are a concise jokester."),
    HumanMessage(content=f"Tell me a short, clean joke about {animal}."),
]




In [9]:
with tracer.start_as_current_span("tool_calling_example"):
    
    ai_msg = tool_model.invoke(
        messages
    )
    
    print(f"Message generated from our llm call")
    rich.print(ai_msg)


    # get tool call and arguments
    tc = ai_msg.tool_calls[0]         
    
    # invoke tool call
    tool_result = animal_joke.invoke(tc["args"] )

    # tool output result
    print(f"tool output result")
    rich.print(tool_result)

Message generated from our llm call


AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 17,
            'prompt_tokens': 73,
            'total_tokens': 90,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4o-mini-2024-07-18',
        'system_fingerprint': 'fp_560af6e559',
        'id': 'chatcmpl-CWTvVINK3k2Eghr9XSGahTVThWcmu',
        'service_tier': 'default',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--21a33aad-c422-461a-805a-be0c0e1b72e8-0',
    tool_calls=[
        {
            'name': 'animal_joke',
            'args': {'animal': 'turtle'},
            'id': 'call_9EJmG8kj9kcKYzx7nPALrW7p',
            'type': 'tool_call'
        }
    ],
    usage_metadata={
        'input_tokens': 73,
        'output_tokens': 17,
        'total_tokens': 90,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

tool output result


Why don’t turtles use public Wi-Fi? Too many shell networks.